In [ ]:
import requests

url = "https://api.languagetool.org/v2/check"
params = {
    "text": "Ths is an exmple sentnce.",
    "language": "en-US"
}

response = requests.post(url, data=params)
print(response.json()["matches"])


In [ ]:
# prompt: load concatenated_text.csv as a pd frame and print its head()

import pandas as pd

# Assuming concatenated_text.csv is in the current working directory
# If not, provide the correct path
try:
  df = pd.read_csv('/content/concatenated_text.csv')
  print(df.head())
except FileNotFoundError:
  print("Error: 'concatenated_text.csv' not found. Please check the file path.")


In [ ]:
pip install pandas requests tqdm


In [ ]:
import pandas as pd
import requests
from tqdm import tqdm

# Function to correct text using LanguageTool API
def correct_text(text):
    url = "https://api.languagetool.org/v2/check"
    params = {"text": text, "language": "en-US"}

    response = requests.post(url, data=params)
    if response.status_code == 200:
        matches = response.json()["matches"]
        for match in matches:
            replacement = match["replacements"][0]["value"] if match["replacements"] else None
            if replacement:
                start, end = match["offset"], match["offset"] + match["length"]
                text = text[:start] + replacement + text[end:]
        return text
    else:
        return text  # Return original text if API call fails

# Load CSV
df = pd.read_csv("/content/concatenated_text.csv")

# Apply spell correction
df["Corrected Text"] = [correct_text(text) if isinstance(text, str) else text for text in tqdm(df["Extracted Text"])]

# Save corrected CSV
df.to_csv("corrected_text.csv", index=False)

print("Spell correction completed! ✅ Check 'corrected_text.csv'")


In [ ]:
!pip install transformers torch accelerate pandas tqdm

In [ ]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm

In [ ]:

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer  # ✅ Correct Import
import torch


In [ ]:
from huggingface_hub import login

login()  # This will prompt you to enter your HF token

In [ ]:

corrector = pipeline("text2text-generation", model="t5-small")
corrected_text = corrector("Correct the spelling: 'thsi is a smaple text'", max_length=50)
print(corrected_text)


In [ ]:
# Step 4: Load the T5 Model (Flan-T5 Base or Large)
model_name = "google/flan-t5-large"  # Change to "flan-t5-base" if limited resources
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")

In [ ]:
# Step 5: Create a text generation pipeline
corrector = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Function to correct spelling errors using Gemma
def correct_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return text  # Skip empty or non-text values

    prompt = f"Craefully Correct the spelling and grammar of this sentence, with minimal changes: '{text}'\nCorrected version:"
    response = corrector(prompt, max_length=100, do_sample=True)

    corrected_text = response[0]["generated_text"].split("Corrected version:")[-1].strip()
    return corrected_text

In [ ]:
csv_path = "/content/concatenated_text.csv"  # Update with your actual file path
main_df = pd.read_csv(csv_path)
sample_df = main_df.head(100).copy()
df = sample_df

In [ ]:
tqdm.pandas()
df["Corrected Text"] = df["Extracted Text"].progress_apply(correct_text)

# Step 8: Save the Corrected CSV File
output_path = "/content/drive/MyDrive/corrected_text.csv"  # Update path if needed
df.to_csv(output_path, index=False)

print("✅ Spell correction completed! Check:", output_path)

In [ ]:
from google.colab import files

output_path = "/content/corrected_text.csv"  # Save in Colab environment
df.to_csv(output_path, index=False)  # Save DataFrame

# Download to your computer
files.download(output_path)